In [1]:
import duckdb
import glob

conn = duckdb.connect("polymarket.db")
conn.execute("PRAGMA threads=2")
conn.execute("PRAGMA enable_object_cache=true")
conn.execute("SET max_memory='2GB'")
conn.execute("SET temp_directory='duckdb_temp/'")

market_path = "../data/polymarket/markets/markets_*.parquet"
trade_path = "../data/polymarket/trades/trades_*.parquet"
block_path = "../data/polymarket/blocks/blocks_*.parquet"

In [ ]:
# drop existing table if it exists
conn.execute("DROP TABLE IF EXISTS trader_counts")

# create the table structure first
conn.execute("""
    CREATE TABLE trader_counts (
        trader VARCHAR,
        trade_count BIGINT
    )
""")

all_files = glob.glob("../data/polymarket/trades/trades_*.parquet")
chunk_size = 100 # process 100 files at a time

print(f"Total files to process: {len(all_files)}")

for i in range(0, len(all_files), chunk_size):
    chunk = all_files[i:i + chunk_size]
    
    # use a temp table to aggregate the chunk first
    conn.execute(f"""
        INSERT INTO trader_counts
        SELECT trader, COUNT(*) 
        FROM read_parquet({chunk})
        CROSS JOIN LATERAL (SELECT maker AS trader UNION ALL SELECT taker)
        WHERE trader IS NOT NULL
        GROUP BY trader
    """)
    print(f"Processed up to file {i + len(chunk)}")

conn.execute("""
    CREATE TABLE final_trader_counts AS 
    SELECT trader, SUM(trade_count) as total_trades
    FROM trader_counts
    GROUP BY trader
""")
conn.execute("DROP TABLE trader_counts")
print("Done!")

Total files to process: 40454
Processed up to file 100
Processed up to file 200
Processed up to file 300
Processed up to file 400
Processed up to file 500
Processed up to file 600
Processed up to file 700
Processed up to file 800
Processed up to file 900
Processed up to file 1000
Processed up to file 1100
Processed up to file 1200
Processed up to file 1300
Processed up to file 1400
Processed up to file 1500
Processed up to file 1600
Processed up to file 1700
Processed up to file 1800
Processed up to file 1900
Processed up to file 2000
Processed up to file 2100
Processed up to file 2200
Processed up to file 2300
Processed up to file 2400
Processed up to file 2500
Processed up to file 2600
Processed up to file 2700
Processed up to file 2800
Processed up to file 2900
Processed up to file 3000
Processed up to file 3100
Processed up to file 3200
Processed up to file 3300
Processed up to file 3400
Processed up to file 3500
Processed up to file 3600
Processed up to file 3700
Processed up to f

In [5]:
conn.execute(f"""SELECT COUNT(*) FROM final_trader_counts WHERE total_trades >= 100""").fetchone()

(479487,)

In [6]:
N = 100 # keep ~1/N of traders
MIN_TRADES = 100

conn.execute("DROP TABLE IF EXISTS traders_sampled")
conn.execute("DROP TABLE IF EXISTS trades_sampled")

# sample 1 / N traders with at least MIN_TRADES trades
conn.execute(f"""
    CREATE TABLE traders_sampled AS
    SELECT trader
    FROM final_trader_counts
    WHERE total_trades >= {MIN_TRADES}
    AND ABS(HASH(trader)) % {N} = 0
""")

# select trades where either maker or taker is in the sampled_traders
conn.execute(f"""
    CREATE TABLE trades_sampled AS
    SELECT *
    FROM '{trade_path}'
    WHERE maker IN (SELECT trader FROM traders_sampled)
    OR taker IN (SELECT trader FROM traders_sampled)
""")

In [8]:
import os

file_path = '../samples/trades_sampled.csv'
if os.path.exists(file_path):
    try:
        os.remove(file_path)
        print("Existing file cleared.")
    except PermissionError:
        print("Warning: File is still locked by another program!")

In [9]:
conn.close()
# Re-establish the connection
conn = duckdb.connect("polymarket.db")

In [14]:
exports = [
    ("trades_sampled", "trades_sampled"),
    ("traders_sampled", "traders_sampled")
]

for table, filename in exports:
    print(f"Exporting {table}...")
    
    # Export to CSV
    conn.execute(f"""
        COPY {table} TO '../samples/{filename}.csv' 
        (HEADER, DELIMITER ',', OVERWRITE)
    """)
    
    # Export to Parquet
    conn.execute(f"""
        COPY {table} TO '../samples/{filename}.parquet' 
        (FORMAT 'parquet', OVERWRITE)
    """)

print("All exports completed successfully.")

Exporting trades_sampled...
Exporting traders_sampled...
All exports completed successfully.


In [11]:
conn.execute("SELECT COUNT(*) FROM trades_sampled").fetchone()

(305708,)

In [12]:
threshold = 0.5

market_query = f"""
CREATE TABLE markets
AS SELECT
    question,
    clob_token_ids ->> '$[0]' AS yes_token,
    clob_token_ids ->> '$[1]' AS no_token,
    CAST(outcome_prices ->> '$[0]' AS FLOAT) AS yes_price,
    CAST(outcome_prices ->> '$[1]' AS FLOAT) AS no_price
FROM '{market_path}';
CREATE INDEX idx_yes_token ON markets(yes_token);
CREATE INDEX idx_no_token ON markets(no_token);
"""

tokens_query = f"""
CREATE TABLE tokens
AS
SELECT
    yes_token AS token,
    CASE
        WHEN yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0} THEN TRUE
        WHEN yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0} THEN FALSE
        ELSE NULL
    END AS implied_outcome,
    CASE
        WHEN (yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0})
          OR (yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0}) THEN TRUE
        ELSE FALSE
    END AS resolved,
    1 = 1 as token_type
FROM markets

UNION ALL

SELECT
    no_token AS token,
    CASE
        WHEN yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0} THEN FALSE
        WHEN yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0} THEN TRUE
        ELSE NULL
    END AS implied_outcome,
    CASE
        WHEN (yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0})
          OR (yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0}) THEN TRUE
        ELSE FALSE
    END AS resolved,
    1 = 0 as token_type
FROM markets;
CREATE INDEX idx_token ON tokens(token);
"""

conn.execute("DROP TABLE IF EXISTS markets")
conn.execute(market_query)
conn.execute("DROP TABLE IF EXISTS tokens")
conn.execute(tokens_query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))